## Traitement données topographiques
### Structure générale

L'ensemble des paramètres est dérivé à partir de l'unique couche Modèle Numérique de Terrain résolution 10m x 10m.
1. Découpage de la couche MNT d'entrée avec les limites du canton pour limiter les temps de calcul
2. Calcul pente et exposition à partir du MNT découpé
3. Calcul de la courbure, concavité/convexité
4. Calcul du TWI (Topographic Wetness Index) avec rééchantillonnage du MNT à plusieurs résolutions
5. Généralisation des variables pour différentes résolutions (exploratoire)
6. Calcul northness à partir de l'exposition
7. Normalisation et clip de toutes les couches (harmonisation spatiale)
8. Resampling des couches à 50m pour homogénéiser (nécessaire pour agrégation finale)
9. Calcul de l'indice Topographique

### 1. Découpage de la frontière vaudoise

In [ ]:
# 1. Découpage par la frontière vaudoise
import rasterio
from rasterio.mask import mask
import geopandas as gpd
import os

# Chemins des fichiers
raster_file = "../data/raw/SwissALTIRegio/swissaltiregio_2056_5728.tif"
vector_file = "../data/raw/Limite Canton Vaud/limites_officielles/OIT.MN95_OIT_TPR_LAD_MO_CANTON.shp"  # ou .shp selon ton fichier
output_file = "../data/processed/Vaud_ALTI10m.tif"

# Créer le dossier de sortie s'il n'existe pas
os.makedirs(os.path.dirname(output_file), exist_ok=True)

# Charger la frontière du canton
gdf = gpd.read_file(vector_file)

# Ouvrir le raster
with rasterio.open(raster_file) as src:
    # Reprojeter la frontière si nécessaire
    if gdf.crs != src.crs:
        gdf = gdf.to_crs(src.crs)

    # Découper le raster
    out_image, out_transform = mask(src, gdf.geometry, crop=True)
    
    # Mettre à jour les métadonnées
    out_meta = src.meta.copy()
    out_meta.update({
        "driver": "GTiff",
        "height": out_image.shape[1],
        "width": out_image.shape[2],
        "transform": out_transform
    })

# Sauvegarder le raster découpé
with rasterio.open(output_file, "w", **out_meta) as dest:
    dest.write(out_image)

print(f"Raster découpé selon le canton de Vaud sauvegardé dans {output_file}")


### 2. Calcul pente et exposition

In [ ]:
# 2. Calcul pente et exposition 10m avec GDAL
from osgeo import gdal
import os

# === Fichiers ===
mnt_10m = "../data/processed/Vaud_ALTI10m.tif"  # <-- ton MNT 10 m découpé
slope_output = "../data/processed/Vaud_slope_10m.tif"
aspect_output = "../data/processed/Vaud_aspect_10m.tif"

# === Vérification que le fichier source existe ===
if not os.path.exists(mnt_10m):
    raise FileNotFoundError(f"❌ Fichier introuvable : {mnt_10m}")

# === Calcul de la pente ===
print("🟢 Calcul de la pente (en degrés)...")
gdal.DEMProcessing(
    slope_output,       # fichier de sortie
    mnt_10m,            # MNT en entrée
    "slope",            # type de calcul
    format="GTiff",
    computeEdges=True,  # étendre le calcul jusqu'aux bords
    slopeFormat="degree"  # ou "percent" si tu veux une pente en %
)
print(f"✅ Pente sauvegardée : {slope_output}")

# === Calcul de l’exposition ===
print("🟢 Calcul de l’exposition (aspect)...")
gdal.DEMProcessing(
    aspect_output,
    mnt_10m,
    "aspect",
    format="GTiff",
    computeEdges=True
)
print(f"✅ Exposition sauvegardée : {aspect_output}")

# === Résumé ===
print("🎉 Terminé ! Les fichiers slope et aspect (10 m) ont été générés avec succès.")


### 3. Calcul de courbure, convexité/concavité

In [ ]:
# 3. Calcul de courbure convexité/concavité 10m avec GDAL
# curvature_gaussian_scale.py
import numpy as np
from osgeo import gdal
from scipy.ndimage import gaussian_filter


# --- INPUT / OUTPUT ---
mnt_path = "../data/processed/Vaud_ALTI10m.tif"
out_curv = "../data/processed/Curvature_res10m/Vaud_curvature_1000m.tif"

# --- Paramètre d'échelle ---
# resolution = 10 m/pixel (ton MNT). Choisis l'échelle physique souhaitée S (mètres)
# puis sigma_pixels = S / resolution.
resolution = 10.0  # m/pixel
S = 1000.0           # échelle d'analyse en mètres (ex : 30 m)
sigma = S / resolution  # sigma en pixels

print(f"Using sigma = {sigma:.2f} pixels (scale {S} m)")

# --- Lecture du MNT ---
ds = gdal.Open(mnt_path, gdal.GA_ReadOnly)
if ds is None:
    raise FileNotFoundError(mnt_path)

band = ds.GetRasterBand(1)
elev = band.ReadAsArray().astype(np.float32)
gt = ds.GetGeoTransform()
proj = ds.GetProjection()
cols = ds.RasterXSize
rows = ds.RasterYSize
cellx = gt[1]
celly = abs(gt[5])

print(f"Raster size: {cols} x {rows}, resolution approx {cellx} m")

# --- Option: mask nodata to NaN, keep mask for output ---
nodata = band.GetNoDataValue()
if nodata is not None:
    mask = (elev == nodata)
    elev = np.where(mask, np.nan, elev)

# --- Appliquer gaussian derivative filters ---
# gaussian_filter handles NaNs poorly; fill NaNs by local interpolation or mask them out.
# Simple approach: replace NaN by local mean (fast) — better methods exist if needed.
if np.isnan(elev).any():
    # replace NaN by global median as quick hack (or use inpaint)
    med = np.nanmedian(elev)
    elev_filled = np.where(np.isnan(elev), med, elev)
else:
    elev_filled = elev

# compute second derivatives (order=(2,0) -> d2/dx2, (0,2) -> d2/dy2)
d2x = gaussian_filter(elev_filled, sigma=sigma, order=(2, 0), mode='mirror') / (cellx**2)
d2y = gaussian_filter(elev_filled, sigma=sigma, order=(0, 2), mode='mirror') / (celly**2)

# curvature (planform / laplacian-like)
curv = d2x + d2y

# re-apply mask nodata as desired
if nodata is not None:
    curv = np.where(mask, nodata, curv)

# --- Sauvegarde geotiff ---
driver = gdal.GetDriverByName("GTiff")
out_ds = driver.Create(out_curv, cols, rows, 1, gdal.GDT_Float32, options=["COMPRESS=LZW", "TILED=YES"])
out_ds.SetGeoTransform(gt)
out_ds.SetProjection(proj)
out_band = out_ds.GetRasterBand(1)
if nodata is not None:
    out_band.SetNoDataValue(nodata)
out_band.WriteArray(curv.astype(np.float32))
out_band.FlushCache()
out_ds = None
ds = None

print("Done — curvature saved to:", out_curv)
### Lecture de la courbure
#Valeur < 0 = convexe, crête, sommet, colline
#Valeur > 0 = cuvette, vallée
#Valeur ~= 0 = plat

### 4. Topographic Wetness Index

In [ ]:
# 4. Calcul de l'indice Topographic Wetness Index (TWI) sans richdem
import numpy as np
import rasterio
from rasterio.windows import Window
from tqdm import tqdm
from scipy.ndimage import uniform_filter
import os

# === Fichiers d'entrée ===
dem_path = "../data/processed/Vaud_ALTI10m.tif"
slope_path = "../data/processed/Vaud_slope_10m.tif"
curv_path = "../data/processed/Vaud_curvature_10m.tif"
output_dir = "../data/processed/TWI/"
os.makedirs(output_dir, exist_ok=True)

# === Paramètres ===
block_size = 1000
epsilon = 1e-6
flow_windows = [10]

# === Lecture du MNT pour récupérer profil + infos géo ===
with rasterio.open(slope_path) as src:
    rows, cols = src.height, src.width
    cellsize = src.res[0]
    transform = src.transform
    crs = src.crs

# === Fonction de lecture bloc avec gestion NaN ===
def read_raster_block(raster_path, y_off, height):
    with rasterio.open(raster_path) as src:
        data = src.read(1, window=Window(0, y_off, src.width, height)).astype(np.float32)
        nodata = src.nodata
        if nodata is not None:
            data[data == nodata] = np.nan
        return data, src.width

# === Fonction de calcul TWI ===
def twi_block(slope, curvature, flow_window_cells):
    slope = np.clip(slope, epsilon, np.pi/2)  # empêche division et valeurs folles
    weights = 1 + np.tanh(curvature)
    acc = uniform_filter(weights, size=flow_window_cells, mode="reflect")
    twi = np.log(acc / np.sin(slope))
    twi[~np.isfinite(twi)] = np.nan
    return twi

# === Boucle sur les tailles de fenêtres ===
for fw_m in flow_windows:
    flow_window_cells = max(1, int(fw_m / cellsize))
    out_path = os.path.join(output_dir, f"TWI_{fw_m}m.tif")

    profile = {
        "driver": "GTiff",
        "height": rows,
        "width": cols,
        "count": 1,
        "dtype": "float32",
        "crs": crs,
        "transform": transform,
        "nodata": -9999,
        "compress": "deflate"
    }

    with rasterio.open(out_path, "w", **profile) as dst:
        for y in tqdm(range(0, rows, block_size), desc=f"Calcul TWI {fw_m}m"):
            h = min(block_size, rows - y)
            slope_block, _ = read_raster_block(slope_path, y, h)
            curv_block, _ = read_raster_block(curv_path, y, h)
            twi = twi_block(slope_block, curv_block, flow_window_cells)
            twi = np.where(np.isnan(twi), -9999, np.clip(twi, -20, 20))
            dst.write(twi.astype(np.float32), 1, window=Window(0, y, slope_block.shape[1], h))

    print(f"✅ TWI calculé et sauvegardé : {out_path}")




### Rééchantillonnage du MNT pour TWI

In [ ]:
# Rééchantillonnage du MNT à différentes résolutions
import rasterio
from rasterio.enums import Resampling
import os

# === Paramètres ===
input_dem = "../data/processed/Vaud_ALTI10m.tif"  # ton MNT original à 10m
output_dir = "../data/processed/MNT_resampled/"
resolutions = [30]  # en mètres
os.makedirs(output_dir, exist_ok=True)

# === Lecture du raster original ===
with rasterio.open(input_dem) as src:
    dem_data = src.read(1, masked=True)
    transform = src.transform
    crs = src.crs
    res_orig = src.res[0]

    for res in resolutions:
        scale = res / res_orig
        new_height = int(src.height / scale)
        new_width = int(src.width / scale)

        # Nouveau profil
        new_transform = rasterio.Affine(transform.a * scale, transform.b, transform.c,
                                        transform.d, transform.e * scale, transform.f)
        profile = src.profile.copy()
        profile.update({
            "height": new_height,
            "width": new_width,
            "transform": new_transform,
            "dtype": "float32",
            "compress": "deflate"
        })

        # === Rééchantillonnage par moyenne ===
        data_resampled = src.read(
            out_shape=(1, new_height, new_width),
            resampling=Resampling.average
        )[0].astype("float32")

        out_path = os.path.join(output_dir, f"DEM_{res}m.tif")
        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(data_resampled, 1)

        print(f"✅ MNT rééchantillonné à {res} m sauvegardé : {out_path}")


### 5. Généralisation des variables pour différentes résolutions (exploratoire)

In [ ]:
# Calcul pente et exposition pour plusieurs résolutions avec GDAL
from osgeo import gdal
import os

# === Répertoires ===
mnt_dir = "../data/processed/MNT_resampled/"   # dossier contenant les MNT rééchantillonnés
output_dir = "../data/processed/"
os.makedirs(output_dir, exist_ok=True)

# === Liste des résolutions à traiter ===
resolutions = [30]

# === Boucle sur les résolutions ===
for res in resolutions:
    if res == 10:
        mnt_path = "../data/processed/Vaud_ALTI10m.tif"
    else:
        mnt_path = os.path.join(mnt_dir, f"DEM_{res}m.tif")

    if not os.path.exists(mnt_path):
        print(f"⚠️ MNT introuvable pour {res} m : {mnt_path}")
        continue

    slope_output = os.path.join(output_dir, f"Vaud_slope_{res}m.tif")
    aspect_output = os.path.join(output_dir, f"Vaud_aspect_{res}m.tif")

    print(f"🟢 Calcul de la pente ({res} m)...")
    gdal.DEMProcessing(
        slope_output,
        mnt_path,
        "slope",
        format="GTiff",
        computeEdges=True,
        slopeFormat="degree"
    )
    print(f"✅ Pente sauvegardée : {slope_output}")

    print(f"🟢 Calcul de l’exposition ({res} m)...")
    gdal.DEMProcessing(
        aspect_output,
        mnt_path,
        "aspect",
        format="GTiff",
        computeEdges=True
    )
    print(f"✅ Exposition sauvegardée : {aspect_output}")

print("🎉 Terminé ! Pentes et expositions générées pour toutes les résolutions.")


In [ ]:
#Calcul de courbure convexité/concavité pour plusieurs résolutions avec GDAL
import numpy as np
from osgeo import gdal
from scipy.ndimage import gaussian_filter
import os

# === Dossiers ===
mnt_dir = "../data/processed/MNT_resampled/"
output_dir = "../data/processed/Curvature_VarRes/"
os.makedirs(output_dir, exist_ok=True)

# === Liste des résolutions à traiter (en mètres) ===
resolutions = [10,30,50,100,300,500,1000]

# === Paramètre d’échelle physique (S, en mètres) ===
# -> correspond à la taille des formes du relief que tu veux capturer
S = 10.0  # par défaut même échelle que la résolution de base, tu peux ajuster

for res in resolutions:
    if res == 10:
        mnt_path = "../data/processed/Vaud_ALTI10m.tif"
    else:
        mnt_path = os.path.join(mnt_dir, f"DEM_{res}m.tif")

    if not os.path.exists(mnt_path):
        print(f"⚠️ MNT introuvable pour {res} m : {mnt_path}")
        continue

    out_curv = os.path.join(output_dir, f"Vaud_curvature_{res}m.tif")

    print(f"🟢 Calcul de la courbure ({res} m)...")

    # --- Lecture du MNT ---
    ds = gdal.Open(mnt_path, gdal.GA_ReadOnly)
    band = ds.GetRasterBand(1)
    elev = band.ReadAsArray().astype(np.float32)
    gt = ds.GetGeoTransform()
    proj = ds.GetProjection()
    cols, rows = ds.RasterXSize, ds.RasterYSize
    cellx, celly = gt[1], abs(gt[5])

    nodata = band.GetNoDataValue()
    if nodata is not None:
        mask = (elev == nodata)
        elev = np.where(mask, np.nan, elev)
    else:
        mask = np.zeros_like(elev, dtype=bool)

    # --- Appliquer un filtre gaussien dérivé ---
    sigma = S / res  # conversion en pixels
    if sigma < 0.5:
        sigma = 0.5  # évite les artefacts si S < res
    print(f"  → sigma = {sigma:.2f} pixels")

    elev_filled = np.where(np.isnan(elev), np.nanmedian(elev), elev)

    # dérivées secondes
    d2x = gaussian_filter(elev_filled, sigma=sigma, order=(2, 0), mode='mirror') / (cellx**2)
    d2y = gaussian_filter(elev_filled, sigma=sigma, order=(0, 2), mode='mirror') / (celly**2)

    curv = d2x + d2y
    if nodata is not None:
        curv = np.where(mask, nodata, curv)

    # --- Sauvegarde GeoTIFF ---
    driver = gdal.GetDriverByName("GTiff")
    out_ds = driver.Create(out_curv, cols, rows, 1, gdal.GDT_Float32,
                           options=["COMPRESS=LZW", "TILED=YES"])
    out_ds.SetGeoTransform(gt)
    out_ds.SetProjection(proj)
    out_band = out_ds.GetRasterBand(1)
    if nodata is not None:
        out_band.SetNoDataValue(nodata)
    out_band.WriteArray(curv.astype(np.float32))
    out_band.FlushCache()
    out_ds = None
    ds = None

    print(f"✅ Courbure sauvegardée : {out_curv}")

print("🎉 Terminé ! Courbures générées pour toutes les résolutions.")


In [ ]:
# Calcul du Topographic Wetness Index (TWI) pour plusieurs résolutions

import numpy as np
import rasterio
from rasterio.windows import Window
from tqdm import tqdm
from scipy.ndimage import uniform_filter
import os

# === Répertoires ===
base_dir = "../data/processed/"
output_dir = os.path.join(base_dir, "TWI/")
os.makedirs(output_dir, exist_ok=True)

# === Fichiers d'entrée explicites ===
resolutions = [30]

input_files = {
    res: {
        "slope": os.path.join(base_dir, f"Vaud_slope_{res}m.tif"),
        "curv": os.path.join(base_dir, f"Vaud_curvature_{res}m.tif"),
    }
    for res in resolutions
}

# === Paramètres ===
block_size = 1000
epsilon = 1e-6
flow_windows = [10]  # en mètres (taille physique de la fenêtre)

# === Fonctions ===
def read_raster_block(raster_path, y_off, height):
    """Lecture par bloc avec gestion du nodata"""
    with rasterio.open(raster_path) as src:
        data = src.read(1, window=Window(0, y_off, src.width, height)).astype(np.float32)
        nodata = src.nodata
        if nodata is not None:
            data[data == nodata] = np.nan
        return data, src.width

def twi_block(slope, curvature, flow_window_cells):
    """Calcul local du TWI à partir des blocs de pente et courbure"""
    slope = np.clip(slope, epsilon, np.pi/2)
    weights = 1 + np.tanh(curvature)
    acc = uniform_filter(weights, size=flow_window_cells, mode="reflect")
    twi = np.log(acc / np.sin(slope))
    twi[~np.isfinite(twi)] = np.nan
    return twi

# === Calcul du TWI pour chaque résolution ===
for res, paths in input_files.items():
    slope_path = paths["slope"]
    curv_path = paths["curv"]

    if not os.path.exists(slope_path) or not os.path.exists(curv_path):
        print(f"⚠️ Fichiers manquants pour {res} m, passage au suivant.")
        continue

    print(f"\n🟢 Calcul du TWI pour la résolution {res} m")

    with rasterio.open(slope_path) as src:
        rows, cols = src.height, src.width
        cellsize = src.res[0]
        transform = src.transform
        crs = src.crs

    for fw_m in flow_windows:
        flow_window_cells = max(1, int(fw_m / cellsize))
        out_path = os.path.join(output_dir, f"TWI_{res}m_fw{fw_m}m.tif")

        profile = {
            "driver": "GTiff",
            "height": rows,
            "width": cols,
            "count": 1,
            "dtype": "float32",
            "crs": crs,
            "transform": transform,
            "nodata": -9999,
            "compress": "deflate"
        }

        with rasterio.open(out_path, "w", **profile) as dst:
            for y in tqdm(range(0, rows, block_size), desc=f"TWI {res}m"):
                h = min(block_size, rows - y)
                slope_block, _ = read_raster_block(slope_path, y, h)
                curv_block, _ = read_raster_block(curv_path, y, h)
                twi = twi_block(slope_block, curv_block, flow_window_cells)
                twi = np.where(np.isnan(twi), -9999, np.clip(twi, -20, 20))
                dst.write(twi.astype(np.float32), 1, window=Window(0, y, slope_block.shape[1], h))

        print(f"✅ TWI {res}m sauvegardé : {out_path}")

print("\n🎉 Tous les TWI multi-résolutions ont été calculés avec succès.")


### 6. Calcul de la Northness à partir de l'aspect

In [ ]:
# Calcul de la northness à partir de l’aspect
import rasterio
import numpy as np
from rasterio import Affine
from rasterio.enums import Resampling

# --- Fichiers ---
aspect_path = "../data/processed/Aspect/Vaud_aspect_50m.tif"
northness_path = "../data/processed/Aspect/Vaud_northness_50m.tif"

# --- Lecture du raster d’aspect ---
with rasterio.open(aspect_path) as src:
    aspect = src.read(1, masked=True)  # lecture en tant que masque (gère les no-data)
    profile = src.profile

# --- Conversion en radians si nécessaire ---
# QGIS ou GDAL donnent souvent l’aspect en degrés (0–360)
aspect_rad = np.deg2rad(aspect)

# --- Calcul de la northness ---
northness = np.cos(aspect_rad)

# --- Mise à jour du profil pour sauvegarde ---
profile.update(
    dtype=rasterio.float32,
    count=1,
    compress='lzw'
)

# --- Sauvegarde du raster northness ---
with rasterio.open(northness_path, 'w', **profile) as dst:
    dst.write(northness.astype(rasterio.float32), 1)

print("✅ Raster de northness créé :", northness_path)


In [ ]:
# Northess: effacement des artéfacts
import rasterio
import numpy as np
import os

# === Fichiers ===
northness_path = "../data/processed/Aspect/Vaud_northness_50m.tif"
slope_path = "../data/processed/Slope/Vaud_slope_50m.tif"
output_path = "../data/processed/Aspect/Vaud_northness_1degmasked0_50m.tif"

# === Paramètre ===
slope_threshold = 1  # degré minimum pour considérer l'aspect/northness comme valide

# === Lecture des rasters ===
with rasterio.open(northness_path) as north_src, rasterio.open(slope_path) as slope_src:
    north = north_src.read(1).astype(np.float32)
    slope = slope_src.read(1).astype(np.float32)
    profile = north_src.profile.copy()

# === Masquage des zones plates ===
# Si la pente est trop faible, on met la valeur à NaN ou NoData
north_masked = np.where(slope >= slope_threshold, north, 0)

# === Sauvegarde ===
profile.update(dtype=rasterio.float32, nodata=np.nan, compress='deflate')
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with rasterio.open(output_path, "w", **profile) as dst:
    dst.write(north_masked, 1)

print(f"✅ Northness masquée sauvegardée : {output_path}")


In [ ]:
# Remapping northness from [-1, 1] to [0, 1]
import rasterio
import numpy as np

# --- INPUT / OUTPUT ---
in_raster = "../data/processed/Aspect/Vaud_northness_1degmasked0_50m.tif"     # northness ∈ [-1, 1]
out_raster = "../data/processed/Aspect/Vaud_northness_50m_1degmask0_01remap.tif"    # remapped to [0, 1]

# --- PROCESS ---
with rasterio.open(in_raster) as src:
    profile = src.profile
    arr = src.read(1).astype("float32")

# remap [-1,1] → [0,1]
# value in [-1,1] → (value + 1) / 2
remapped = (arr + 1.0) / 2.0

# clamp numerical drift (rare but safe)
remapped = np.clip(remapped, 0.0, 1.0)

# --- EXPORT ---
profile.update(dtype="float32")

with rasterio.open(out_raster, "w", **profile) as dst:
    dst.write(remapped, 1)

print("Remap complete →", out_raster)


### 7. Normalisation et clip de toutes les couches (harmonisation spatiale)

In [ ]:
# Normalisation avancée avec clipping percentile et options
import os
import json
import numpy as np
import rasterio
from rasterio.enums import Resampling

def normalize_with_clip(input_path, output_path, *,
                        pmin=5.0, pmax=95.0,
                        prelog=False,
                        symmetric=False,
                        nodata_keep=True,
                        nodata_value=None,
                        overwrite=False):
    """
    Normalise un raster avec clipping percentile et min-max.
    - symmetric=True : pour variables centrées sur 0 (ex: courbure). On calcule le percentile pmax
      de la distribution des valeurs absolues puis on clip à +/- that_value.
    - prelog=True : applique np.log1p avant clipping (utile pour TWI)
    - pmin,pmax : percentiles (0..100) ; if symmetric only pmax used.
    - nodata_keep : conserve nodata (remplacé par np.nan pendant calcul).
    """
    if os.path.exists(output_path) and not overwrite:
        raise FileExistsError(f"{output_path} exists. Use overwrite=True to replace.")

    with rasterio.open(input_path) as src:
        profile = src.profile.copy()
        arr = src.read(1).astype(np.float32)
        src_nodata = src.nodata if nodata_value is None else nodata_value

    # mask nodata
    if src_nodata is not None:
        mask = (arr == src_nodata) | (~np.isfinite(arr))
    else:
        mask = ~np.isfinite(arr)
    valid = arr[~mask]

    if valid.size == 0:
        raise ValueError("No valid pixels found in input raster.")

    # optionally apply pre-log (on valid values only)
    if prelog:
        valid_proc = np.log1p(valid)
    else:
        valid_proc = valid

    # symmetric clipping around 0 for centered variables
    if symmetric:
        # use percentile on absolute values
        abs_vals = np.abs(valid_proc)
        cutoff = np.percentile(abs_vals, pmax)
        low_val = -cutoff
        high_val = cutoff
        print(f"[symmetric] cutoff (abs) @ p{pmax} = {cutoff:.6g}")
    else:
        low_val = np.percentile(valid_proc, pmin)
        high_val = np.percentile(valid_proc, pmax)
        print(f"percentiles: p{pmin}={low_val:.6g}, p{pmax}={high_val:.6g}")

    # Create working copy (float) and replace nodata with nan
    work = arr.astype(np.float32)
    work[mask] = np.nan

    # if prelog, transform entire working array's valid pixels
    if prelog:
        with np.errstate(invalid='ignore'):
            work = np.where(np.isfinite(work), np.log1p(work), work)

    # clip
    work_clipped = np.clip(work, low_val, high_val)

    # min-max normalize
    denom = (high_val - low_val)
    if denom == 0:
        # constant field after clipping
        norm = np.full_like(work_clipped, np.nan, dtype=np.float32)
        norm[np.isfinite(work_clipped)] = 0.5
    else:
        norm = (work_clipped - low_val) / denom
    norm[mask] = profile.get("nodata", np.nan) if nodata_keep else np.nan

    # update profile
    profile.update(dtype=rasterio.float32, compress='deflate')
    if nodata_keep:
        profile.update(nodata=profile.get("nodata", -9999))
        out_nodata = profile["nodata"]
    else:
        out_nodata = np.nan

    # ensure output dir
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # write raster
    with rasterio.open(output_path, "w", **profile) as dst:
        write_arr = np.where(np.isfinite(norm), norm, out_nodata).astype(np.float32)
        dst.write(write_arr, 1)

    # save metadata about normalization
    meta = {
        "input": os.path.basename(input_path),
        "output": os.path.basename(output_path),
        "pmin": pmin,
        "pmax": pmax,
        "prelog": bool(prelog),
        "symmetric": bool(symmetric),
        "low_val": float(low_val),
        "high_val": float(high_val),
        "nodata_kept": bool(nodata_keep),
    }
    meta_path = os.path.splitext(output_path)[0] + "_meta.json"
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    print(f"Saved normalized raster: {output_path}")
    print(f"Saved metadata: {meta_path}")
    return output_path, meta_path


In [ ]:
# Courbure
normalize_with_clip("../data/processed/Curvature_res10m_ConvVar/Vaud_curvature_500mConv.tif",
                    "../data/processed/Curvature_res10m_ConvVar/Vaud_curvature_500mConv_normclip1090.tif",
                    pmin=15, pmax=85, prelog=False, symmetric=False, overwrite=True)
print("Courbure Fait.")

# TWI
normalize_with_clip("../data/processed/TWI/TWI_50m.tif",
                    "../data/processed/TWI/TWI_50m_normclip_0286.tif",
                    pmin=2, pmax=86, prelog=False, symmetric=False, overwrite=True)
print("TWI Fait.")

# slope
normalize_with_clip("../data/processed/Slope/Vaud_slope_1000m.tif",
                    "../data/processed/Slope/Vaud_slope_1000m_normclip_0298.tif",
                    pmin=2, pmax=98, prelog=False, symmetric=False, overwrite=True)
print("Pente Fait.")

# MNT
normalize_with_clip("../data/processed/MNT_resampled/DEM_50m.tif",
                    "../data/processed/MNT_resampled/DEM_50m_normclip_0298.tif",
                    pmin=2, pmax=98, prelog=False, symmetric=False, overwrite=True)
print("MNT Fait.")

# Topo_index_final
normalize_with_clip(r"../data/processed/Vaud_topo_index_50m.tif",
                    r"../data/processed/Vaud_topo_index_50m_normclip0295.tif",
                    pmin=2, pmax=95, prelog=False, symmetric=False, overwrite=True)
print("Topo_index_final Fait.")

### 8. Resampling des couches à 50m pour homogénéiser (nécessaire pour agrégation finale)

In [ ]:
# Resampling raster à résolution 50m
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject, calculate_default_transform

def resample_to_50m(input_path, output_path, target_res=50):
    with rasterio.open(input_path) as src:
        
        # compute transform and output shape
        transform, width, height = calculate_default_transform(
            src.crs,
            src.crs,
            src.width,
            src.height,
            src.bounds.left,
            src.bounds.bottom,
            src.bounds.right,
            src.bounds.top,
            resolution=target_res
        )

        profile = src.profile.copy()
        profile.update({
            "transform": transform,
            "width": width,
            "height": height
        })

        with rasterio.open(output_path, "w", **profile) as dst:
            
            # --- RESAMPLE DATA ---
            reproject(
                source=rasterio.band(src, 1),
                destination=rasterio.band(dst, 1),
                src_transform = src.transform,
                src_crs = src.crs,
                dst_transform = transform,
                dst_crs = src.crs,
                resampling = Resampling.average   # PERFECT for downsampling curvature
            )

    print(f"✔ Resampled saved → {output_path}")


In [ ]:
files = [
    "../data/processed/Curvature_res10m_ConvVar/Vaud_curvature_30mConv_normclip1090.tif",
    "../data/processed/Curvature_res10m_ConvVar/Vaud_curvature_50mConv_normclip1090.tif",  # si déjà 50m, ça copiera / ajustera
    "../data/processed/Curvature_res10m_ConvVar/Vaud_curvature_100mConv_normclip1090.tif",
    "../data/processed/Curvature_res10m_ConvVar/Vaud_curvature_300mConv_normclip1090.tif",
]

for f in files:
    out = f.replace(".tif", "_res50m.tif")
    resample_to_50m(f, out)

### 9. Calcul de l'indice Topographique

In [ ]:
# Calcul de l'indice topographique
import numpy as np
import rasterio
from tqdm import tqdm

# -------------------------
# 1. PARAMÈTRES UTILISATEUR
# -------------------------

# chemins vers les rasters (déjà normalisés et clipés)
rasters = {
    "slope":      r"../data/processed/Slope/Vaud_slope_50m_normclip_0298.tif",
    "twi":        r"../data/processed/TWI/TWI_50m_normclip_1585.tif",
    "northness":  r"../data/processed/Aspect/Vaud_northness_50m_1degmask0_01remap.tif",
    "altitude":   r"../data/processed/MNT_resampled/DEM_50m_normclip_0298.tif",
    "curv30":     r"../data/processed/Curvature_res10m_ConvVar/Vaud_curvature_30mConv_normclip1090_res50m.tif",
    "curv50":     r"../data/processed/Curvature_res10m_ConvVar/Vaud_curvature_50mConv_normclip1090_res50m.tif",
    "curv100":    r"../data/processed/Curvature_res10m_ConvVar/Vaud_curvature_100mConv_normclip1090_res50m.tif",
    "curv300":    r"../data/processed/Curvature_res10m_ConvVar/Vaud_curvature_300mConv_normclip1090_res50m.tif",
}

# pondérations (normalisées automatiquement)
weights = {
    "slope":    1,
    "twi":      2,
    "northness":1,
    "altitude": 0.5,
    "curv30":   0.3,
    "curv50":   0.5,
    "curv100":  0.5,
    "curv300":  0.2,
}

# variables à inverser hydrologiquement (haut = sec)
invert = {
    "slope":     False,
    "twi":       True,
    "northness": True,
    "altitude":  False,
    "curv30":    False,
    "curv50":    False,
    "curv100":   False,
    "curv300":   False,
}

output_path = r"../data/processed/Topo_index_50m.tif"

# -------------------------
# 2. LIRE LES RASTERS
# -------------------------
arrays = {}
profiles = {}

for key, path in rasters.items():
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32)
        nodata = src.nodata
        if nodata is not None:
            arr[arr == nodata] = np.nan
        arrays[key] = arr
        profiles[key] = src.profile

# -------------------------
# 3. ALIGNER LES RASTERS
# -------------------------
ref_shape = next(iter(arrays.values())).shape
for key in arrays:
    arr = arrays[key]
    if arr.shape != ref_shape:
        arrays[key] = arr[:ref_shape[0], :ref_shape[1]]

# -------------------------
# 4. INVERSION DES RASTERS
# -------------------------
for key in arrays:
    if invert.get(key, False):
        arrays[key] = 1 - arrays[key]

# -------------------------
# 5. CALCUL DE L'INDICE TOPO
# -------------------------
weight_vec = np.array([weights[k] for k in arrays.keys()], dtype=float)
weight_vec = weight_vec / np.sum(weight_vec)

index = np.zeros(ref_shape, dtype=np.float32)

for i, key in enumerate(tqdm(arrays.keys(), desc="Combinaison")):
    index += arrays[key] * weight_vec[i]

# -------------------------
# 6. EXPORT
# -------------------------
profile_out = profiles[next(iter(arrays.keys()))].copy()
profile_out.update(dtype=rasterio.float32, count=1, compress="deflate", nodata=-9999)

with rasterio.open(output_path, "w", **profile_out) as dst:
    dst.write(np.where(np.isnan(index), -9999, index).astype(np.float32), 1)

print(f"✅ Indice topographique sauvegardé : {output_path}")


In [ ]:
# renormalisation de l'indice topo final
# il y a besoin, car moyenne pondérée ne sera jamais entre 0 et 1.
# cela permet d'augmenter le contraste final.
normalize_with_clip(input_path=r"../data/processed/Topo_index_50m.tif",
                    output_path=r"../data/processed/Topo_index_50m_normalized.tif",
                    pmin=0.1,pmax=99.9, overwrite=True)